# Gikomba Market Stall Performance Analysis

This notebook uses core Python to analyze Wanjiku's 14 days of sales and stock data.

## Dataset

The following cell contains the sales, day, and stock data provided for the assignment.

In [1]:
daily_sales = [2300, 1950, 2800, 2100, 3600, 5200, 4100,
               2500, 2200, 3100, 2400, 3900, 5600, 4700]   # 14 days, KES
days = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"] * 2


# qty = kg in stock, reorder = reorder level (kg), cost/price = KES per kg
stock = {
    "Tomatoes": {"qty": 18, "reorder": 25, "cost": 60, "price": 90},
    "Onions":   {"qty": 40, "reorder": 20, "cost": 70, "price": 100},
    "Sukuma":   {"qty": 9,  "reorder": 15, "cost": 30, "price": 50},
    "Potatoes": {"qty": 55, "reorder": 30, "cost": 50, "price": 75},
    "Avocado":  {"qty": 12, "reorder": 10, "cost": 80, "price": 140},
}


## Task 1: Summarize 14-day sales

I calculate the total, average, highest, and lowest sales, then use each value's list position to identify its day and week.

In [2]:
total_sales = sum(daily_sales)
avg_sales = total_sales / len(daily_sales)
highest_sale = max(daily_sales)
lowest_sale = min(daily_sales)

best_idx = daily_sales.index(highest_sale)
worst_idx = daily_sales.index(lowest_sale)
best_week = 1 if best_idx < 7 else 2
worst_week = 1 if worst_idx < 7 else 2

print(f"Total Sales: KES {total_sales:,}")
print(f"Average Daily Sales: KES {avg_sales:,.2f}")
print(f"Highest Sale: KES {highest_sale:,} on Week {best_week} {days[best_idx]}")
print(f"Lowest Sale: KES {lowest_sale:,} on Week {worst_week} {days[worst_idx]}")


Total Sales: KES 46,450
Average Daily Sales: KES 3,317.86
Highest Sale: KES 5,600 on Week 2 Sat
Lowest Sale: KES 1,950 on Week 1 Tue


## Task 2: Calculate weekly totals and growth

The function uses slicing to total the first and second seven-day periods, then I compare the two totals as a percentage.

In [3]:
def week_totals(sales):
    """Return a tuple containing the totals for week 1 and week 2."""
    return sum(sales[:7]), sum(sales[7:])


w1_total, w2_total = week_totals(daily_sales)
growth_pct = ((w2_total - w1_total) / w1_total) * 100

print(f"Week 1 Total: KES {w1_total:,}")
print(f"Week 2 Total: KES {w2_total:,}")
print(f"Week-on-Week Growth: {growth_pct:.1f}%")


Week 1 Total: KES 22,050
Week 2 Total: KES 24,400
Week-on-Week Growth: 10.7%


## Task 3: Classify daily performance

Each day is classified against the overall average: more than 20% above average is Strong, more than 20% below is Weak, and the rest is Normal. The reusable `classify_sales()` function accepts both the sales list and average explicitly, then uses `classify_day()` for each amount.


In [4]:
def classify_day(amount, average):
    """Classify a sale as Strong, Normal, or Weak relative to the average."""
    if amount > average * 1.2:
        return "Strong"
    if amount < average * 0.8:
        return "Weak"
    return "Normal"


def classify_sales(sales, average):
    """Count Strong, Normal, and Weak days for a sales list."""
    counts = {}
    for sale in sales:
        category = classify_day(sale, average)
        counts[category] = counts.get(category, 0) + 1
    return counts


day_counts = classify_sales(daily_sales, avg_sales)
print("Day Classification Counts:", day_counts)


Day Classification Counts: {'Weak': 6, 'Normal': 4, 'Strong': 4}


## Task 4: Compare weekdays and weekends

I separate the two weekday and weekend groups from both weeks, calculate their averages, and express the difference relative to weekday sales.

In [5]:
weekdays = [daily_sales[i] for i in range(len(daily_sales)) if i % 7 < 5]
weekends = [daily_sales[i] for i in range(len(daily_sales)) if i % 7 >= 5]

avg_weekday = sum(weekdays) / len(weekdays)
avg_weekend = sum(weekends) / len(weekends)
diff_pct = ((avg_weekend - avg_weekday) / avg_weekday) * 100

print(f"Average Weekday Sales: KES {avg_weekday:,.2f}")
print(f"Average Weekend Sales: KES {avg_weekend:,.2f}")
print(f"Weekend vs Weekday Difference: {diff_pct:.1f}% higher")


Average Weekday Sales: KES 2,685.00
Average Weekend Sales: KES 4,900.00
Weekend vs Weekday Difference: 82.5% higher


## Task 5: Identify restocking needs and gross margins

The reorder function loops through the stock dictionary and returns products at or below their reorder levels. I also calculate each product's gross margin percentage.

In [6]:
def products_to_reorder(stock_data):
    """Return product names whose quantities are at or below reorder levels."""
    return [
        product
        for product, info in stock_data.items()
        if info["qty"] <= info["reorder"]
    ]


gross_margins = {}
for product, info in stock.items():
    gross_margins[product] = (info["price"] - info["cost"]) / info["price"] * 100
    print(f"{product} Gross Margin: {gross_margins[product]:.1f}%")

items_to_restock = products_to_reorder(stock)
print("Products to reorder:", items_to_restock)


Tomatoes Gross Margin: 33.3%
Onions Gross Margin: 30.0%
Sukuma Gross Margin: 40.0%
Potatoes Gross Margin: 33.3%
Avocado Gross Margin: 42.9%
Products to reorder: ['Tomatoes', 'Sukuma']


## Task 6: Print the Stall Performance Report

I combine the key results in a formatted report and use f-string formatting to display KES amounts with thousands separators.

In [7]:
print("=" * 44)
print("        STALL PERFORMANCE REPORT")
print("=" * 44)
print(f"Total Sales:          KES {total_sales:,}")
print(f"Average Daily Sales:  KES {avg_sales:,.2f}")
print(f"Highest Sale:         KES {highest_sale:,}")
print(f"Lowest Sale:          KES {lowest_sale:,}")
print(f"Week 1 Total:         KES {w1_total:,}")
print(f"Week 2 Total:         KES {w2_total:,}")
print(f"Week-on-Week Growth:  {growth_pct:.1f}%")
print(f"Average Weekday:      KES {avg_weekday:,.2f}")
print(f"Average Weekend:      KES {avg_weekend:,.2f}")
print(f"Products to Restock:  {', '.join(items_to_restock)}")
print("=" * 44)


        STALL PERFORMANCE REPORT
Total Sales:          KES 46,450
Average Daily Sales:  KES 3,317.86
Highest Sale:         KES 5,600
Lowest Sale:          KES 1,950
Week 1 Total:         KES 22,050
Week 2 Total:         KES 24,400
Week-on-Week Growth:  10.7%
Average Weekday:      KES 2,685.00
Average Weekend:      KES 4,900.00
Products to Restock:  Tomatoes, Sukuma


## Task 7: Turn the analysis into business actions

I use the strongest findings—growth, day-of-week performance, and reorder levels—to make specific recommendations supported by the calculated numbers.

In [8]:
print(f"Growth to build on: {growth_pct:.1f}%")
print(f"Weekday sales opportunity: weekends average {diff_pct:.1f}% more than weekdays")
print(f"Immediate restocking priority: {', '.join(items_to_restock)}")


Growth to build on: 10.7%
Weekday sales opportunity: weekends average 82.5% more than weekdays
Immediate restocking priority: Tomatoes, Sukuma


## Recommendations for Wanjiku

Wanjiku's stall generated **KES 46,450** over 14 days and grew by **10.7%** from Week 1 to Week 2, so she should maintain the practices that supported the increase. Weekend sales averaged **KES 4,900**, which was **82.5% higher** than the weekday average of KES 2,685; she should run weekday promotions or bundle offers to improve slower mid-week demand. She should immediately restock **Tomatoes** and **Sukuma**, because their quantities of 18 kg and 9 kg are at or below reorder levels of 25 kg and 15 kg. She should also monitor Avocado, which has the highest gross margin at **42.9%**, and consider promoting it to improve profitability.